# 第 06 章 · 回合调度：追加输入、等待与取消

模型正在运行时用户补充一句话，应该追加给谁？任务刚要结束时这句话到达，会不会丢失？
调度器的责任是为输入、等待请求和终态指定唯一 owner，而不是把所有异步工作简单 `gather` 起来。

我们在同一个 asyncio 事件循环里制造明确的先后顺序。没有 `await` 的状态转移在本原型里不可被其他协程插入；
外部线程必须投递消息，不能直接修改这些对象。代码使用 Notebook 的顶层 `await`，不需要嵌套 `asyncio.run()`。

**运行边界**：本章使用最终包的共享组件，逐格验证本专题机制；完整任务运行见第 31 章。所有代码使用 Python 标准库；不调用真实模型、不读取凭据、不自动安装软件。按顺序运行；需要磁盘的实验只在 `fresh_run_dir()` 新建目录内写入。修改参数后，以重新执行产生的断言与输出为准。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter


In [2]:
schedule = load_example('06', 'scheduler')
import asyncio

def expect_error(error_type, operation, message):
    try:
        operation()
    except error_type as error:
        assert message in str(error), str(error)
        return str(error)
    raise AssertionError(f'应出现 {error_type.__name__}: {message}')


## 1. 接受输入与消费输入是两次状态变化


    第一条输入占据执行槽，后续输入按 acceptance order 排队。`claim` 是当前回合真正消费队列的边界；
    在它之前，accepted 并不意味着已发送给模型。针对旧 turn 的追加必须拒绝，不能猜测用户想发给新 turn。
    修改 `followup`，观察顺序与归属，而不是只检查列表长度。


In [3]:
followup = '先修复筛选，再运行测试。'
scheduler = schedule.Scheduler('exercise-1')
status, ref = scheduler.start_if_idle('input-1', '修复 task-list')
assert status == 'started'
status2, ref2 = scheduler.start_or_steer('input-2', followup)
assert status2 == 'steered' and ref2 == ref
assert scheduler.active.consumed == []
batch = scheduler.claim(ref)
assert [x.input_id for x in batch] == ['input-1', 'input-2']
assert [x.order for x in batch] == [1, 2]
wrong = schedule.TurnRef(ref.run_id, 'old-turn', 0)
assert scheduler.steer(wrong, 'input-3', '旧回合输入') == 'wrong_turn'
assert 'input-3' not in scheduler.accepted
assert scheduler.begin_finish(ref)
await scheduler.finish(ref)
{'consumed': [(x.input_id, x.text) for x in batch], 'terminal': scheduler.terminal[ref]['status']}


{'consumed': [('input-1', '修复 task-list'), ('input-2', '先修复筛选，再运行测试。')],
 'terminal': 'completed'}

## 2. 结束与追加竞争，必须有明确的提交点


    如果追加先发生，结束操作必须重新发现队列并继续工作；如果关闭入口先发生，新输入应有明确去处。
    本教学策略把晚到输入记为下一轮 deferred，不自动重启旧轮。它是教学取舍，不是 Codex 全部入口的逐字复刻。


In [4]:
s = schedule.Scheduler('exercise-2')
_, r = s.start_if_idle('a', '开始')
s.claim(r)
assert s.steer(r, 'b', '先到的追加') == 'steered'
assert not s.begin_finish(r)
assert [x.input_id for x in s.claim(r)] == ['b']
assert s.begin_finish(r)
late_status, _ = s.start_or_steer('c', '入口关闭后的追加')
assert late_status == 'deferred'
await s.finish(r)
_, next_ref = s.start_if_idle('d', '下一轮启动', previous=r)
next_batch = s.claim(next_ref)
assert [x.input_id for x in next_batch] == ['c', 'd']
assert s.begin_finish(next_ref)
await s.finish(next_ref)
assert len(s.terminal) == 2
{'late_input_receipt': late_status, 'next_turn_consumed': [x.input_id for x in next_batch]}


{'late_input_receipt': 'deferred', 'next_turn_consumed': ['c', 'd']}

## 3. 取消立即释放问题，但等待清理后才发布终态


    pending 问题不应等待慢清理才收到取消，否则 UI 和上层协程会长时间挂住。
    用两个 Event 制造清理屏障：看到 entered 才说明清理已开始；release 控制何时确认完成。
    在屏障之间发送迟到答案，必须被拒绝，且新任务不能抢占尚未释放的执行槽。


In [5]:
class CleanupBarrier:
    def __init__(self, fail=False):
        self.entered = asyncio.Event()
        self.release = asyncio.Event()
        self.fail = fail
        self.calls = 0
    async def close_and_join(self, owner, cancel):
        self.calls += 1
        self.entered.set()
        await self.release.wait()
        return {'cleanup': 'unknown' if self.fail else 'confirmed'}
s = schedule.Scheduler('exercise-3')
_, r = s.start_if_idle('start', '等待选择')
s.claim(r)
barrier = CleanupBarrier()
s.attach_participant(r, barrier)
question = s.question(r, 'question-1')
cancelling = asyncio.create_task(s.interrupt(r))
await asyncio.wait_for(barrier.entered.wait(), 1)
pending_result = await s.pending.wait(question)
assert pending_result['status'] == 'cancelled'
assert not s.answer(r, 'question-1', '迟到的同意')
assert s.start_if_idle('too-early', '新任务')[0] == 'not_idle'
assert r not in s.terminal
barrier.release.set()
await asyncio.wait_for(cancelling, 1)
assert barrier.calls == 1 and s.terminal[r]['status'] == 'cancelled'
assert len([e for e in s.events if e[0] == 'turn_terminal']) == 1
{'question': pending_result['status'], 'turn': s.terminal[r]['status'], 'cleanup_calls': barrier.calls}


{'question': 'cancelled', 'turn': 'cancelled', 'cleanup_calls': 1}

## 4. 等待者取消不能毁掉共享 Future


    多个观察者可能等待同一个用户问题。取消其中一个观察者，不应把问题本身取消。
    `asyncio.shield` 保护底层 Future，pending owner 才能决定它的唯一终态。
    再用实际定时器观察 timeout，确认计时器收尾后重复答案不会改变既定结果。


In [6]:
pending = schedule.PendingRequests()
owner = schedule.TurnRef('r', 't', 1)
request = pending.request('shared', owner)
observer = asyncio.create_task(pending.wait(request))
await asyncio.sleep(0)
observer.cancel()
try:
    await observer
except asyncio.CancelledError:
    pass
assert not request.future.cancelled()
assert pending.settle('shared', owner, 'answered', '继续')
assert (await pending.wait(request))['answer'] == '继续'
timeout_seconds = 0.02
timed = pending.request('timed', owner, timeout=timeout_seconds)
timed_result = await asyncio.wait_for(pending.wait(timed), 1)
assert timed_result['status'] == 'timeout'
assert not pending.settle('timed', owner, 'answered', '太迟')
assert timed.timer.cancelled()
{'shared_future_result': request.future.result()['status'], 'timer_result': timed_result['status']}


{'shared_future_result': 'answered', 'timer_result': 'timeout'}

## 5. 清理不确定时，容量不能自动开放


    “取消请求已发送”与“资源清理确认”不是一回事。未知清理若仍释放执行槽，新任务可能与旧进程同时写工作区。
    本例注入一个返回 unknown 的清理参与者：调度器可以提交明确的失败终态，但保留占用，要求外部核对。
    这是状态机故障注入，没有假称存在或清理了真实子进程。


In [7]:
s = schedule.Scheduler('exercise-5')
_, r = s.start_if_idle('start', '任务')
s.claim(r)
uncertain = CleanupBarrier(fail=True)
uncertain.release.set()
s.attach_participant(r, uncertain)
await s.interrupt(r)
assert s.terminal[r]['status'] == 'cleanup_unknown'
assert s.active.phase == 'blocked_cleanup'
assert s.start_if_idle('new', '不能启动')[0] == 'not_idle'
assert not s.observe_result(r, {'late': True})
{'terminal': s.terminal[r]['status'], 'slot': s.active.phase, 'late_event': s.events[-1][0]}


{'terminal': 'cleanup_unknown',
 'slot': 'blocked_cleanup',
 'late_event': 'late_result'}

## 回到 Codex 源码

以下仅陈述固定提交 `c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee` 的静态源码事实；本 Notebook 没有执行 Rust 测试。Python 类型、字段、预算与简化策略属于教学设计，不能据此声称 Codex 使用同一实现。

- [start_if_idle](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn_input.rs#L540-L632)：StartIfIdle 在 active_turn 锁下检查占用、expected previous，并预留 ActiveTurn；准备失败清理预留。

- [Session::steer_input](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/session/turn_input.rs#L961-L1054)：steer_input 检查 active task、expected ID、任务类型、非空及 schema；保存 metadata 并分配 acceptance_order 后注入队列。

- [Session::on_task_finished](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tasks/mod.rs#L717-L803)：on_task_finished 在同一 active lock 下做末次 pending 检查及 task.take；可继续则返回采样要求，否则设置 finishing 并记录剩余输入。


## 现在应能解释什么


    能解释输入接受、输入消费、入口关闭、清理完成和终态提交为何分开；能指出 pending request 和 turn 的唯一 owner。
    接回 Loop 时复用原有执行器和取消令牌，只让调度器拥有回合准入与终态，不重写模型或工具执行链。

    本章验证单事件循环局部状态机和真实 asyncio 定时/取消行为；没有验证跨线程、跨进程、真实 UI 或完整 Agent 回合。
